# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Aleeza-Maryam/Aleeza-ML-Internship-WEEK1/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

# Research Question

Can observable search and content signals be used to rank pages that are potential content-refresh opportunities better than a simple hand-written rule?

## Lane

Refresh / Content Opportunity Scoring

## Decision Supported

The goal is to help content teams prioritize which pages should be reviewed first for a possible content refresh.

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

## Data

This project uses the FlyRank Internship Warehouse dataset hosted on Hugging Face.

### Tables

The main table is `fact_content_daily_performance`, which contains daily search and analytics performance for content items.

The `dim_content` table is used for content-level metadata such as content age, content type, search volume, and word count.

### Development Window

The main development data uses a mid-panel month such as March 2026.

The final month is treated as a sealed test period rather than being used to develop the label or rule.

### Excluded Data

Client names, domains, URLs, private queries, credentials, and other identifying information are excluded.

Future-window information is also excluded from the features because it would cause data leakage.

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

## Methodology

The analysis will use observable signals that are available at the decision moment.

Candidate features include search impressions, clicks, average search position, analytics activity, content age, days since last update, search volume, and word count.

A future-window outcome will be used as the prediction target so that the model learns from information available before the outcome occurs.

The existing hand-written refresh rule from Week 4 will be used as the baseline.

The machine-learning model will be evaluated against this baseline using the same validation split.

Leakage checks will ensure that future performance information and label-derived variables are not used as model features.

In [1]:
import os
import duckdb
from google.colab import userdata

# Load Hugging Face token from Colab Secrets
HF_TOKEN = userdata.get("HF_TOKEN")

print("Token loaded:", HF_TOKEN is not None)

# Hugging Face dataset
HF_DATASET = "hf://datasets/FlyRank/internship-warehouse"

# Create DuckDB connection
con = duckdb.connect()

# Configure Hugging Face authentication
con.execute(f"""
    CREATE OR REPLACE SECRET hf_secret (
        TYPE HUGGINGFACE,
        TOKEN '{HF_TOKEN}'
    )
""")

print("DuckDB connection ready.")
print("HF dataset:", HF_DATASET)

Token loaded: True
DuckDB connection ready.
HF dataset: hf://datasets/FlyRank/internship-warehouse


In [2]:
q = con.sql(f"""
SELECT
    COUNT(*) AS rows,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM read_parquet(
    '{HF_DATASET}/fact_content_daily_performance/month=2026-03/data_0.parquet'
)
""")

q.show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────┬────────────┬────────────┐
│  rows   │ first_date │ last_date  │
│  int64  │    date    │    date    │
├─────────┼────────────┼────────────┤
│ 9841378 │ 2026-03-01 │ 2026-03-31 │
└─────────┴────────────┴────────────┘



In [ ]:
features_df = march_features.df()

print("Rows:", len(features_df))
print("Columns:", features_df.columns.tolist())

features_df.head(10)

In [15]:
features_df.head(10)

In [ ]:
content_features = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    AVG(gsc_impressions) AS impressions,
    AVG(gsc_clicks) AS clicks,
    AVG(gsc_avg_position) AS avg_position,
    AVG(ga4_pageviews) AS pageviews,
    AVG(ga4_engaged_sessions) AS engaged_sessions
FROM read_parquet(
    '{HF_DATASET}/fact_content_daily_performance/month=2026-03/data_0.parquet'
)
GROUP BY
    client_hash_id,
    content_hash_id
LIMIT 10000
""")

features_df = content_features.df()
print("Performance features:", features_df.shape)

In [4]:
content_schema = con.sql(f"""
DESCRIBE
SELECT *
FROM read_parquet(
    '{HF_DATASET}/dim_content.parquet'
)
""")

content_schema.show()

┌────────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│        column_name         │ column_type │  null   │   key   │ default │  extra  │
│          varchar           │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ client_hash_id             │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ keyword_hash_id            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ url_hash_id                │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ keyword_char_count         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ keyword_token_count        │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ url_char_count             │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ content_created_date       │ DATE        │ YES     │ NULL    │ 

In [6]:
march_features = con.sql(f"""
WITH performance AS (
    SELECT
        client_hash_id,
        content_hash_id,
        AVG(gsc_impressions) AS impressions,
        AVG(gsc_clicks) AS clicks,
        AVG(gsc_avg_position) AS avg_position,
        AVG(ga4_pageviews) AS pageviews,
        AVG(ga4_engaged_sessions) AS engaged_sessions
    FROM read_parquet(
        '{HF_DATASET}/fact_content_daily_performance/month=2026-03/data_0.parquet'
    )
    GROUP BY
        client_hash_id,
        content_hash_id
    LIMIT 10000
),

content_info AS (
    SELECT
        client_hash_id,
        content_hash_id,
        content_created_date,
        content_updated_date,
        content_type,
        search_volume,
        word_count,
        is_published,
        is_deleted
    FROM read_parquet(
        '{HF_DATASET}/dim_content.parquet'
    )
)

SELECT
    p.*,
    c.content_created_date,
    c.content_updated_date,
    c.content_type,
    c.search_volume,
    c.word_count,
    c.is_published,
    c.is_deleted,

    -- Content age in days
    DATE_DIFF(
        'day',
        c.content_created_date,
        DATE '2026-03-31'
    ) AS content_age_days,

    -- Days since last update
    DATE_DIFF(
        'day',
        c.content_updated_date,
        DATE '2026-03-31'
    ) AS days_since_update

FROM performance p

LEFT JOIN content_info c
ON p.client_hash_id = c.client_hash_id
AND p.content_hash_id = c.content_hash_id
""")

features_df = march_features.df()

print("Final feature dataset shape:", features_df.shape)

features_df.head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Final feature dataset shape: (10000, 16)


,client_hash_id,content_hash_id,impressions,clicks,avg_position,pageviews,engaged_sessions,content_created_date,content_updated_date,content_type,search_volume,word_count,is_published,is_deleted,content_age_days,days_since_update
0,client_625b6439094e23e4,content_01853f98190c1687,0.0,0.0,NaN,0.0,0.0,2025-07-09,2026-05-20,feedly article,<NA>,<NA>,True,False,265,-50
1,client_625b6439094e23e4,content_01aa5e52698d5a05,0.0,0.0,NaN,0.0,0.0,2024-11-28,2026-05-20,keyword article,<NA>,<NA>,True,False,488,-50
2,client_625b6439094e23e4,content_01c3f88bf62e0066,0.0,0.0,NaN,0.0,0.0,2024-11-28,2026-05-20,keyword article,<NA>,<NA>,True,False,488,-50
3,client_625b6439094e23e4,content_01c43dae6ed0ba22,0.0,0.0,NaN,0.0,0.0,2025-07-25,2026-05-20,feedly article,<NA>,<NA>,True,False,249,-50
4,client_625b6439094e23e4,content_01f37eda9c89a0a9,0.0,0.0,NaN,0.0,0.0,2025-07-17,2026-05-20,feedly article,<NA>,<NA>,True,False,257,-50
5,client_625b6439094e23e4,content_01fd5d93047977d3,0.0,0.0,NaN,0.0,0.0,2025-08-19,2026-05-20,feedly article,<NA>,<NA>,True,False,224,-50
6,client_625b6439094e23e4,content_024d8260fd31c874,0.0,0.0,NaN,0.0,0.0,2024-11-29,2025-09-17,keyword article,<NA>,<NA>,True,False,487,195
7,client_625b6439094e23e4,content_0255a46005f5f302,0.0,0.0,NaN,0.0,0.0,2025-09-29,2026-05-20,feedly article,<NA>,<NA>,True,False,183,-50
8,client_625b6439094e23e4,content_025eb3a961a81d40,0.0,0.0,NaN,0.0,0.0,2025-07-13,2026-05-20,feedly article,<NA>,<NA>,True,False,261,-50
9,client_625b6439094e23e4,content_025fdb40dcb5999d,0.0,0.0,NaN,0.0,0.0,2025-07-23,2026-05-20,feedly article,<NA>,<NA>,True,False,251,-50


In [8]:
import pandas as pd

# Check how many rows have dates after our decision date
decision_date = pd.Timestamp("2026-03-31")

print("Rows with future content_updated_date:")

future_updates = (
    pd.to_datetime(features_df["content_updated_date"]) > decision_date
).sum()

print(future_updates)

print("\nRows with future content_created_date:")

future_created = (
    pd.to_datetime(features_df["content_created_date"]) > decision_date
).sum()

print(future_created)

Rows with future content_updated_date:
9726

Rows with future content_created_date:
0


In [9]:
import pandas as pd
import numpy as np

# Make a clean copy
clean_df = features_df.copy()

# Convert dates
clean_df["content_created_date"] = pd.to_datetime(
    clean_df["content_created_date"]
)

# Decision date
decision_date = pd.Timestamp("2026-03-31")

# Recalculate content age safely
clean_df["content_age_days"] = (
    decision_date - clean_df["content_created_date"]
).dt.days

# Remove the unsafe future-derived feature
if "days_since_update" in clean_df.columns:
    clean_df = clean_df.drop(columns=["days_since_update"])

# Also remove future update date from modeling features
# Keep it out of the feature set to avoid leakage
print("Clean dataset shape:", clean_df.shape)

print("\nContent age summary:")
print(clean_df["content_age_days"].describe())

print("\nRemaining columns:")
print(clean_df.columns.tolist())

Clean dataset shape: (10000, 15)

Content age summary:
count    10000.000000
mean       317.302300
std        112.454586
min         34.000000
25%        260.000000
50%        368.000000
75%        393.000000
max        488.000000
Name: content_age_days, dtype: float64

Remaining columns:
['client_hash_id', 'content_hash_id', 'impressions', 'clicks', 'avg_position', 'pageviews', 'engaged_sessions', 'content_created_date', 'content_updated_date', 'content_type', 'search_volume', 'word_count', 'is_published', 'is_deleted', 'content_age_days']


In [10]:
# Check whether April 2026 data is available

april_check = con.sql(f"""
SELECT
    COUNT(*) AS rows,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM read_parquet(
    '{HF_DATASET}/fact_content_daily_performance/month=2026-04/data_0.parquet'
)
""")

april_check.show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────┬────────────┬────────────┐
│   rows   │ first_date │ last_date  │
│  int64   │    date    │    date    │
├──────────┼────────────┼────────────┤
│ 10424730 │ 2026-04-01 │ 2026-04-30 │
└──────────┴────────────┴────────────┘



In [11]:
# Create an honest March -> April outcome dataset

march_april = con.sql(f"""
WITH march_performance AS (
    SELECT
        client_hash_id,
        content_hash_id,
        AVG(gsc_impressions) AS march_impressions,
        AVG(gsc_clicks) AS march_clicks,
        AVG(gsc_avg_position) AS march_avg_position,
        AVG(ga4_pageviews) AS march_pageviews,
        AVG(ga4_engaged_sessions) AS march_engaged_sessions
    FROM read_parquet(
        '{HF_DATASET}/fact_content_daily_performance/month=2026-03/data_0.parquet'
    )
    GROUP BY
        client_hash_id,
        content_hash_id
    LIMIT 10000
),

april_performance AS (
    SELECT
        client_hash_id,
        content_hash_id,
        AVG(gsc_impressions) AS april_impressions,
        AVG(gsc_clicks) AS april_clicks,
        AVG(gsc_avg_position) AS april_avg_position
    FROM read_parquet(
        '{HF_DATASET}/fact_content_daily_performance/month=2026-04/data_0.parquet'
    )
    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT
    m.*,
    a.april_impressions,
    a.april_clicks,
    a.april_avg_position,

    CASE
        WHEN m.march_impressions > 0
        THEN (
            a.april_impressions - m.march_impressions
        ) / m.march_impressions
        ELSE NULL
    END AS future_impression_change

FROM march_performance m
INNER JOIN april_performance a
ON m.client_hash_id = a.client_hash_id
AND m.content_hash_id = a.content_hash_id
""")

march_april_df = march_april.df()

print("March-April dataset shape:", march_april_df.shape)

march_april_df.head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

March-April dataset shape: (10000, 11)


,client_hash_id,content_hash_id,march_impressions,march_clicks,march_avg_position,march_pageviews,march_engaged_sessions,april_impressions,april_clicks,april_avg_position,future_impression_change
0,client_9958f0a7ae1df715,content_f3d06b55e2fc0544,14.290323,0.129032,14.915032,0.161290,0.0,6.033333,0.033333,18.550993,-0.577803
1,client_9958f0a7ae1df715,content_0d41e54e1bb47c54,1.741935,0.000000,32.442424,0.000000,0.0,0.666667,0.000000,42.035714,-0.617284
2,client_9958f0a7ae1df715,content_c5430ffb35d67070,3.677419,0.000000,80.502585,0.000000,0.0,3.966667,0.000000,67.810256,0.078655
3,client_9958f0a7ae1df715,content_643eb4ca4be16e27,3.354839,0.000000,25.184665,0.000000,0.0,1.966667,0.000000,39.791333,-0.413782
4,client_9958f0a7ae1df715,content_80a23b2fbb67bc69,0.709677,0.000000,75.753030,0.000000,0.0,0.400000,0.000000,79.138889,-0.436364
5,client_9958f0a7ae1df715,content_1434ccc73da4ae48,16.903226,0.000000,31.427819,0.096774,0.0,3.800000,0.000000,63.635332,-0.775191
6,client_9958f0a7ae1df715,content_6dc717eb28c67a11,0.290323,0.000000,7.285714,0.064516,0.0,0.000000,0.000000,NaN,-1.000000
7,client_9958f0a7ae1df715,content_dccee1781c4d74ee,5.741935,0.000000,37.544141,0.000000,0.0,5.200000,0.033333,48.093433,-0.094382
8,client_9958f0a7ae1df715,content_0f80781b0447052d,5.419355,0.000000,47.296690,0.000000,0.0,3.833333,0.000000,60.920101,-0.292659
9,client_9958f0a7ae1df715,content_0d5f604ed84bbe79,1.645161,0.032258,30.514739,0.032258,0.0,0.600000,0.000000,69.547619,-0.635294


In [12]:
# Check the distribution of future performance change

print(
    march_april_df["future_impression_change"]
    .describe()
)

print("\nPercentiles:")

print(
    march_april_df["future_impression_change"]
    .quantile([0.10, 0.25, 0.50, 0.75, 0.90])
)

count    6079.000000
mean        0.040703
std         3.095268
min        -1.000000
25%        -0.564060
50%        -0.268598
75%         0.114946
max       198.433333
Name: future_impression_change, dtype: float64

Percentiles:
0.10   -0.748291
0.25   -0.564060
0.50   -0.268598
0.75    0.114946
0.90    0.722222
Name: future_impression_change, dtype: float64


In [14]:
# Keep only rows where the future outcome is available
model_df = march_april_df.dropna(
    subset=["future_impression_change"]
).copy()

# Create the future outcome label
model_df["future_decline_label"] = (
    model_df["future_impression_change"] <= -0.30
).astype(int)

print("Model dataset shape:", model_df.shape)

print("\nLabel counts:")
print(model_df["future_decline_label"].value_counts())

print("\nDecline rate:")
print(
    round(
        model_df["future_decline_label"].mean(),
        3
    )
)

Model dataset shape: (6079, 12)

Label counts:
future_decline_label
0    3205
1    2874
Name: count, dtype: int64

Decline rate:
0.473


In [15]:
# Check whether position is associated with future decline

model_df["position_bucket"] = pd.cut(
    model_df["march_avg_position"],
    bins=[0, 3, 10, 20, 50, float("inf")],
    labels=["1-3", "4-10", "11-20", "21-50", "51+"]
)

position_check = (
    model_df
    .groupby("position_bucket", observed=True)
    .agg(
        n=("future_decline_label", "size"),
        decline_rate=("future_decline_label", "mean"),
        avg_future_change=("future_impression_change", "mean")
    )
)

print(position_check)

                    n  decline_rate  avg_future_change
position_bucket                                       
1-3               261      0.360153           0.346256
4-10             2281      0.430075           0.172299
11-20            1427      0.540995          -0.144737
21-50            1574      0.520330          -0.109606
51+               526      0.395437           0.033466


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

## Results

This section will compare the machine-learning model with the hand-written baseline on the same validation split.

The final comparison will report the selected ranking metric and include supporting charts.

## 5. Limitations

*What this work cannot claim.*

## Limitations

This analysis identifies patterns and ranking opportunities rather than causal effects.

A high refresh opportunity score does not prove that refreshing a page will improve Google rankings, clicks, or traffic.

The model is intended as decision support for prioritizing human review, not as an automatic publishing or content-removal system.

The analysis is also limited by the available warehouse fields, data coverage, and the chosen time windows.

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

## Ranked Recommendations

The final model will produce a ranked list of content items with an opportunity score, reason code, and recommended action.

The recommendations will be used to prioritize human review for possible content refreshes.

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

## Artifacts

The final paper will include:

- Model versus baseline performance
- Model evaluation metrics
- Feature importance or model interpretation
- Ranked recommendation examples
- Supporting charts

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
